# Tutorial: Order Workflows

Audience:
- Developers implementing entry-fill and scale-in behavior.

Prerequisites:
- Familiarity with `Order` and portfolio `Position` concepts.

Learning goals:
- Fill an entry order and create a linked position.
- Inspect auto-linked exit orders.
- Apply a scale-in fill to an existing position.


## Outline

1. Setup imports
2. Create and normalize an entry order
3. Fill entry order and inspect outputs
4. Scale into the position


In [ ]:
from swing_screener.execution import Order
from swing_screener.execution.order_workflows import (
    fill_entry_order,
    normalize_orders,
    scale_in_fill,
)


## Step 1 - Start from empty order and position collections

Create a pending entry order, then normalize orders to ensure metadata is consistent.


In [ ]:
orders = []
positions = []

entry_order = Order(
    order_id="ORD-ENTRY-001",
    ticker="AAPL",
    status="pending",
    order_type="BUY_LIMIT",
    quantity=100,
    limit_price=175.00,
    order_date="2024-01-15",
    order_kind="entry",
)
orders.append(entry_order)

orders, updated = normalize_orders(orders)
print(f"Normalized orders: {updated}")
print(f"Entry order id: {entry_order.order_id}")


## Step 2 - Fill the entry order

This updates the entry order and creates both a position and linked exit orders.


In [ ]:
orders, positions = fill_entry_order(
    orders=orders,
    positions=positions,
    order_id="ORD-ENTRY-001",
    fill_price=175.50,
    fill_date="2024-01-16",
    quantity=100,
    stop_price=170.00,
    tp_price=182.00,
    fee_eur=1.50,
)

print(f"Orders after fill: {len(orders)}")
print(f"Positions after fill: {len(positions)}")


In [ ]:
for o in orders:
    print(f"{o.order_id}: {o.status} {o.order_type} {o.ticker}")

for p in positions:
    print(f"Position: {p.ticker} {p.shares} shares @ ${p.entry_price}")
    print(f"  Stop: ${p.stop_price}, Position ID: {p.position_id}")

exit_orders = [o for o in orders if o.order_kind in ("stop", "take_profit")]
print(f"Auto-linked exit orders: {len(exit_orders)}")
for o in exit_orders:
    print(f"{o.order_id}: {o.order_type} @ ${o.stop_price or o.limit_price}")


## Step 3 - Scale into the position

Add another pending entry order and fill it with `scale_in_fill`.


In [ ]:
scale_order = Order(
    order_id="ORD-ENTRY-002",
    ticker="AAPL",
    status="pending",
    order_type="BUY_LIMIT",
    quantity=50,
    limit_price=174.00,
    order_date="2024-01-17",
    order_kind="entry",
)
orders.append(scale_order)

orders, positions = scale_in_fill(
    orders=orders,
    positions=positions,
    order_id="ORD-ENTRY-002",
    fill_price=174.00,
    fill_date="2024-01-18",
    quantity=50,
    fee_eur=1.00,
)

print(f"Orders after scale-in: {len(orders)}")
print(f"Positions after scale-in: {len(positions)}")
for p in positions:
    print(f"Position: {p.ticker} {p.shares} shares @ ${p.entry_price}")
